# ResNet-8 MNIST Training & Export for Raspberry Pi Pico
Train a ResNet-8 on MNIST, export Q8.8 weights as a C header file.

**Requirements:** `pip install torch torchvision numpy matplotlib`

In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
import textwrap
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

print('PyTorch:', torch.__version__)
print('Device :', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ── Cell 2: Config ───────────────────────────────────────────────────────────
# Change these if needed
EPOCHS      = 15
BATCH_SIZE  = 128
LR          = 1e-3
SAVE_PTH    = 'resnet_mnist.pth'       # where to save trained weights
OUT_HEADER  = 'weights_mnist.h'        # output C header file
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

# Must match resnet_pico.c defines
INPUT_C     = 1
NUM_CLASSES = 10
L0_OUT_C    = 16
L1_C        = 16
L2_IN_C     = 16
L2_OUT_C    = 32
L3_C        = 32

print(f'Training on: {DEVICE}')
print(f'Epochs: {EPOCHS}  Batch: {BATCH_SIZE}  LR: {LR}')

In [ ]:
# ── Cell 3: Model Definition ─────────────────────────────────────────────────
class ResBlock(nn.Module):
    """Identity residual block — same spatial size and channel count."""
    def __init__(self, channels):
        super().__init__()
        self.conv_a = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn_a   = nn.BatchNorm2d(channels)
        self.conv_b = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn_b   = nn.BatchNorm2d(channels)

    def forward(self, x):
        out = F.relu(self.bn_a(self.conv_a(x)))
        out = self.bn_b(self.conv_b(out))
        return F.relu(out + x)            # skip connection


class ResBlockDS(nn.Module):
    """Downsampling residual block — stride=2, channels double, projection shortcut."""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv_a = nn.Conv2d(in_c,  out_c, 3, stride=2, padding=1, bias=False)
        self.bn_a   = nn.BatchNorm2d(out_c)
        self.conv_b = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn_b   = nn.BatchNorm2d(out_c)
        self.proj   = nn.Conv2d(in_c,  out_c, 1, stride=2, bias=False)  # 1x1 projection
        self.bn_p   = nn.BatchNorm2d(out_c)

    def forward(self, x):
        shortcut = self.bn_p(self.proj(x))
        out = F.relu(self.bn_a(self.conv_a(x)))
        out = self.bn_b(self.conv_b(out))
        return F.relu(out + shortcut)     # skip connection with projection


class PicoResNet(nn.Module):
    """
    ResNet-8 for 32x32 greyscale input, 10 classes.
    Matches resnet_pico.c layer-for-layer.
    """
    def __init__(self):
        super().__init__()
        self.conv0 = nn.Conv2d(INPUT_C, L0_OUT_C, 3, padding=1, bias=True)
        self.bn0   = nn.BatchNorm2d(L0_OUT_C)
        self.rb1   = ResBlock(L1_C)                  # 32x32x16
        self.rb2   = ResBlockDS(L2_IN_C, L2_OUT_C)  # 32x32x16 → 16x16x32
        self.rb3   = ResBlock(L3_C)                  # 16x16x32
        self.fc    = nn.Linear(L3_C, NUM_CLASSES)

    def forward(self, x):
        x = F.relu(self.bn0(self.conv0(x)))  # 32x32x16
        x = self.rb1(x)                       # 32x32x16
        x = self.rb2(x)                       # 16x16x32
        x = self.rb3(x)                       # 16x16x32
        x = x.mean(dim=[2, 3])                # global avg pool → 32
        return self.fc(x)                     # 10 logits


# Count parameters
model = PicoResNet()
total = sum(p.numel() for p in model.parameters())
print(f'Model created — {total:,} parameters')
print(model)

In [ ]:
# ── Cell 4: Data ─────────────────────────────────────────────────────────────
tf = transforms.Compose([
    transforms.Pad(2),                        # 28x28 → 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST('./data', train=True,  download=True, transform=tf)
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=tf)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds):,} images  |  Test: {len(test_ds):,} images')

# Preview a random batch
imgs, lbls = next(iter(DataLoader(test_ds, batch_size=10, shuffle=True)))
fig, axes = plt.subplots(1, 10, figsize=(20, 3))
for ax, img, lbl in zip(axes, imgs, lbls):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(lbl.item(), fontsize=14)
    ax.axis('off')
plt.suptitle('Random sample from test set (re-run cell to shuffle)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 5: Training ─────────────────────────────────────────────────────────
model     = PicoResNet().to(DEVICE)
opt       = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn   = nn.CrossEntropyLoss()

train_losses, train_accs, test_accs = [], [], []

def evaluate(dl):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            n += len(y)
    return correct / n

print(f'Training on {DEVICE} for {EPOCHS} epochs...\n')
print(f'{"Epoch":>6}  {"Loss":>8}  {"Train Acc":>10}  {"Test Acc":>10}')
print('-' * 42)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, n = 0.0, 0, 0

    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out  = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        total_loss += loss.item() * len(y)
        correct    += (out.argmax(1) == y).sum().item()
        n          += len(y)

    scheduler.step()

    avg_loss  = total_loss / n
    train_acc = correct / n
    test_acc  = evaluate(test_dl)

    train_losses.append(avg_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f'{epoch:>6}  {avg_loss:>8.4f}  {train_acc*100:>9.2f}%  {test_acc*100:>9.2f}%')

# Save weights
torch.save(model.state_dict(), SAVE_PTH)
print(f'\nWeights saved → {SAVE_PTH}')

In [ ]:
# ── Cell 6: Training Curves ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, EPOCHS+1), train_losses, 'o-', color='#7F77DD', linewidth=2, label='Train loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(range(1, EPOCHS+1), [a*100 for a in train_accs], 'o-', color='#1D9E75', linewidth=2, label='Train acc')
ax2.plot(range(1, EPOCHS+1), [a*100 for a in test_accs],  's-', color='#D85A30', linewidth=2, label='Test acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(90, 100)

plt.suptitle('ResNet-8 Training on MNIST', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Final test accuracy: {test_accs[-1]*100:.2f}%')

In [ ]:
# ── Cell 7: Sanity Check + Visualise Predictions ─────────────────────────────
model.eval()

# Grab 10 random test images
imgs, lbls = next(iter(DataLoader(test_ds, batch_size=10, shuffle=True)))

with torch.no_grad():
    preds = model(imgs.to(DEVICE)).argmax(1).cpu()

fig, axes = plt.subplots(1, 10, figsize=(20, 3))
for ax, img, lbl, pred in zip(axes, imgs, lbls, preds):
    ax.imshow(img.squeeze(), cmap='gray')
    color = 'green' if pred == lbl else 'red'
    ax.set_title(f'✓ {pred.item()}' if pred==lbl else f'✗\nt:{lbl.item()}\np:{pred.item()}',
                 fontsize=11, color=color)
    ax.axis('off')

plt.suptitle('Model predictions — green=correct  red=wrong  (re-run to shuffle)', fontsize=12)
plt.tight_layout()
plt.show()

# Fixed first-8 sanity check (same as train_and_export.py)
x8, y8 = zip(*[test_ds[i] for i in range(8)])
x8 = torch.stack(x8)
with torch.no_grad():
    p8 = model(x8.to(DEVICE)).argmax(1).cpu().tolist()
print('\nSanity check — first 8 test images (fixed):')
print(f'  Predicted : {p8}')
print(f'  True      : {[y for y in y8]}')

In [ ]:
# ── Cell 8: BN Folding + Weight Extraction ───────────────────────────────────
def np_(t):
    return t.detach().cpu().numpy().astype(np.float32)

def fold_bn(conv_w, conv_b, bn_gamma, bn_beta, bn_mean, bn_var, eps=1e-5):
    """Absorb BatchNorm into conv weights and bias."""
    std    = np.sqrt(bn_var + eps)
    scale  = bn_gamma / std
    w_fold = conv_w * scale[:, None, None, None]
    b_fold = (conv_b - bn_mean) * scale + bn_beta
    return w_fold, b_fold

def extract_weights(m):
    """Extract all weights, BN-folded, transposed to C layout [kH][kW][in_c][out_c]."""
    m = m.cpu().eval()
    w = {}

    # conv0 + bn0
    cw, cb = np_(m.conv0.weight), np_(m.conv0.bias)
    cw, cb = fold_bn(cw, cb, np_(m.bn0.weight), np_(m.bn0.bias),
                     np_(m.bn0.running_mean), np_(m.bn0.running_var))
    w['w_conv0'] = cw.transpose(2, 3, 1, 0)   # [3,3,1,16]
    w['b_conv0'] = cb

    def fold_rb(rb, prefix):
        wa = np_(rb.conv_a.weight); ba = np.zeros(wa.shape[0], np.float32)
        wb = np_(rb.conv_b.weight); bb = np.zeros(wb.shape[0], np.float32)
        wa, ba = fold_bn(wa, ba, np_(rb.bn_a.weight), np_(rb.bn_a.bias),
                         np_(rb.bn_a.running_mean), np_(rb.bn_a.running_var))
        wb, bb = fold_bn(wb, bb, np_(rb.bn_b.weight), np_(rb.bn_b.bias),
                         np_(rb.bn_b.running_mean), np_(rb.bn_b.running_var))
        w[f'w_{prefix}_a'] = wa.transpose(2, 3, 1, 0)
        w[f'b_{prefix}_a'] = ba
        w[f'w_{prefix}_b'] = wb.transpose(2, 3, 1, 0)
        w[f'b_{prefix}_b'] = bb

    fold_rb(m.rb1, 'rb1')
    fold_rb(m.rb3, 'rb3')

    rb2 = m.rb2
    wa = np_(rb2.conv_a.weight); ba = np.zeros(wa.shape[0], np.float32)
    wb = np_(rb2.conv_b.weight); bb = np.zeros(wb.shape[0], np.float32)
    wp = np_(rb2.proj.weight);   bp = np.zeros(wp.shape[0], np.float32)
    wa, ba = fold_bn(wa, ba, np_(rb2.bn_a.weight), np_(rb2.bn_a.bias),
                     np_(rb2.bn_a.running_mean), np_(rb2.bn_a.running_var))
    wb, bb = fold_bn(wb, bb, np_(rb2.bn_b.weight), np_(rb2.bn_b.bias),
                     np_(rb2.bn_b.running_mean), np_(rb2.bn_b.running_var))
    wp, bp = fold_bn(wp, bp, np_(rb2.bn_p.weight), np_(rb2.bn_p.bias),
                     np_(rb2.bn_p.running_mean), np_(rb2.bn_p.running_var))
    w['w_rb2_a']    = wa.transpose(2, 3, 1, 0)
    w['b_rb2_a']    = ba
    w['w_rb2_b']    = wb.transpose(2, 3, 1, 0)
    w['b_rb2_b']    = bb
    w['w_rb2_proj'] = wp.transpose(2, 3, 1, 0)
    w['b_rb2_proj'] = bp

    w['w_fc'] = np_(m.fc.weight).T   # [32, 10]
    w['b_fc'] = np_(m.fc.bias)       # [10]

    return w

weights = extract_weights(model)
print('Extracted weight arrays:')
for k, v in weights.items():
    print(f'  {k:<16} shape={str(v.shape):<20} min={v.min():+.4f}  max={v.max():+.4f}')

In [ ]:
# ── Cell 9: Quantise to Q8.8 ─────────────────────────────────────────────────
def quantise(arr):
    q = np.round(arr * 256).astype(np.int64)
    n_clip = np.sum(np.abs(q) > 32767)
    if n_clip:
        print(f'  WARNING: {n_clip} values clipped to int16 range')
    return np.clip(q, -32768, 32767).astype(np.int16)

# Show quantisation error per array
print('Quantisation error (float32 vs Q8.8):')
total_err = 0
for k, arr in weights.items():
    q    = quantise(arr).astype(np.float32) / 256.0
    err  = np.abs(arr - q).mean()
    total_err += err
    print(f'  {k:<16}  mean_err={err:.6f}')
print(f'\nOverall mean error: {total_err/len(weights):.6f}  (should be < 0.005)')

In [ ]:
# ── Cell 10: Generate C Header ────────────────────────────────────────────────
KEY_ORDER = [
    'w_conv0', 'b_conv0',
    'w_rb1_a', 'b_rb1_a', 'w_rb1_b', 'b_rb1_b',
    'w_rb2_a', 'b_rb2_a', 'w_rb2_b', 'b_rb2_b',
    'w_rb2_proj', 'b_rb2_proj',
    'w_rb3_a', 'b_rb3_a', 'w_rb3_b', 'b_rb3_b',
    'w_fc',   'b_fc',
]

def array_to_c(name, arr, cols=16):
    flat = arr.flatten()
    rows = []
    for i in range(0, len(flat), cols):
        rows.append('    ' + ', '.join(str(int(v)) for v in flat[i:i+cols]) + ',')
    dims = ''.join(f'[{d}]' for d in arr.shape)
    return f'static const int16_t {name}{dims} = {{\n' + '\n'.join(rows) + '\n};\n'

def generate_header(weights):
    parts = [textwrap.dedent("""\
        /*
         * weights_mnist.h  — auto-generated by Jupyter notebook
         * All values are Q8.8 fixed-point (int16_t).
         * Include in resnet_pico.c and remove the zeroed placeholder arrays.
         */
        #pragma once
        #include <stdint.h>
    """)]
    for key in KEY_ORDER:
        arr = weights[key]
        q   = quantise(arr)
        parts.append(f'/* {key}  shape={arr.shape}  min={arr.min():.4f}  max={arr.max():.4f} */')
        parts.append(array_to_c(key, q))
    return '\n'.join(parts)

header = generate_header(weights)

with open(OUT_HEADER, 'w') as f:
    f.write(header)

print(f'Header written → {OUT_HEADER}')
print(f'File size: {len(header):,} characters')
print(f'\nFirst 20 lines preview:')
print('\n'.join(header.split('\n')[:20]))

In [ ]:
# ── Cell 11: Visualise What the Filters Learned ───────────────────────────────
# Show the 16 Conv0 filters as images
w_conv0_float = weights['w_conv0']   # shape [3,3,1,16]

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i, ax in enumerate(axes.flat):
    kernel = w_conv0_float[:, :, 0, i]   # 3x3 filter for channel i
    im = ax.imshow(kernel, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(f'filter {i}', fontsize=10)
    ax.axis('off')

plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6)
plt.suptitle('Conv0 — 16 learned 3×3 filters (blue=negative, red=positive weights)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 12: Next Steps Summary ───────────────────────────────────────────────
print(textwrap.dedent(f"""
    ═══════════════════════════════════════════════════════
    ✓  Training complete
    ✓  Final test accuracy : {test_accs[-1]*100:.2f}%
    ✓  Weights saved       : {SAVE_PTH}
    ✓  C header written    : {OUT_HEADER}
    ═══════════════════════════════════════════════════════

    Next steps to flash on your Pico:

    1. Copy {OUT_HEADER} into your Pico project folder

    2. In resnet_pico.c, remove the zeroed weight arrays
       and add at the top:
           #include "weights_mnist.h"

    3. Rebuild:
           cmake --build build

    4. Flash resnet_pico.uf2 to your Pico

    5. Connect with:
           screen /dev/ttyACM0 115200   (Linux/macOS)
           PuTTY → Serial → COMx        (Windows)

    6. Type  start  at the prompt
    ═══════════════════════════════════════════════════════
"""))